In [24]:
%pip -q install pandas numpy pyarrow rapidfuzz pydantic scikit-learn

In [25]:
from __future__ import annotations

import hashlib
import logging
import re
import unicodedata
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError
import warnings
warnings.filterwarnings('ignore')

In [26]:
try:
    from IPython.display import display  # type: ignore
except Exception:
    display = print

try:
    from rapidfuzz import fuzz
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("mplads_preprocessing")

In [27]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
DEFAULT_ROOT = Path.cwd()
if (DEFAULT_ROOT / "Datasets").exists():
    PROJECT_ROOT = DEFAULT_ROOT
elif Path("/content/drive/MyDrive/MPLADs").exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/MPLADs")
else:
    PROJECT_ROOT = DEFAULT_ROOT

DATA_DIR = PROJECT_ROOT / "Datasets"
OUTPUT_DIR = PROJECT_ROOT / "shared_preprocessing_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Data dir: %s", DATA_DIR)
logger.info("Output dir: %s", OUTPUT_DIR)

In [35]:
NULL_LIKE = {"", "nan", "none", "null", "n/a", "na", "n.a.", "-", "--"}

HONORIFIC_RE = re.compile(
    r"^(DR|SHRI|SMT|MS|MRS|MR|KUMARI|KUM|KM|PROF|SH)\.?\s+",
    flags=re.IGNORECASE,
)

TRAILING_PARENS_RE = re.compile(r"(?:\s*\([^)]*\)\s*)+$")
NON_ALNUM_RE = re.compile(r"[^A-Z0-9]+")
WS_RE = re.compile(r"\s+")
WORK_ID_RE = re.compile(r"WS/\s*MP\d+/\d{4}-\d{4}/\d+", flags=re.IGNORECASE)


def normalize_whitespace(value: Any) -> Any:
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\xa0", " ").replace("\t", " ")
    text = WS_RE.sub(" ", text).strip()
    return text if text else pd.NA


def clean_text(value: Any) -> Any:
    text = normalize_whitespace(value)
    if pd.isna(text):
        return pd.NA
    return str(text)


def normalize_state_name(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).replace("&", " And ")
    text = WS_RE.sub(" ", text).strip()
    return text.title()


def parse_indian_amount(value: Any) -> float:
    if pd.isna(value):
        return np.nan
    text = clean_text(value)
    if pd.isna(text):
        return np.nan
    s = str(text)
    s = s.replace("Rs.", "").replace("Rs", "")
    s = s.replace(",", "")
    s = s.replace("(", "-").replace(")", "")
    s = re.sub(r"[^0-9.\-]", "", s)
    if s in {"", "-", ".", "-."}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def parse_date(value: Any) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    text = clean_text(value)
    if pd.isna(text):
        return pd.NaT
    return pd.to_datetime(text, format="%d-%b-%Y", errors="coerce")


def normalize_column_name(name: Any) -> str:
    text = unicodedata.normalize("NFKC", str(name))
    text = text.replace("\xa0", " ").replace("\t", " ")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text


def standardize_columns(df: pd.DataFrame, alias_map: Dict[str, str]) -> pd.DataFrame:
    rename_map: Dict[str, str] = {}
    normalized_alias_map = {normalize_column_name(k): v for k, v in alias_map.items()}
    for col in df.columns:
        key = normalize_column_name(col)
        if key in normalized_alias_map:
            rename_map[col] = normalized_alias_map[key]
    return df.rename(columns=rename_map)


def drop_footer_rows(df: pd.DataFrame, pk_col: str) -> pd.DataFrame:
    if pk_col not in df.columns:
        return df.copy()
    out = df.copy()
    out = out.dropna(how="all")
    numeric_pk = pd.to_numeric(out[pk_col], errors="coerce")
    keep = numeric_pk.notna()
    return out.loc[keep].copy()


def first_non_null(series: pd.Series) -> Any:
    for value in series.tolist():
        if pd.notna(value):
            if isinstance(value, str) and value.strip() == "":
                continue
            return value
    return pd.NA


def unique_join(series: pd.Series) -> Any:
    values = []
    for value in series.tolist():
        if pd.notna(value):
            text = str(value).strip()
            if text and text not in values:
                values.append(text)
    if not values:
        return pd.NA
    return " | ".join(values)


def token_sort_text(text: str) -> str:
    tokens = re.findall(r"[A-Z0-9]+", text.upper())
    tokens.sort()
    return " ".join(tokens)


def token_sort_ratio(a: str, b: str) -> int:
    if HAS_RAPIDFUZZ:
        return int(fuzz.token_sort_ratio(a, b))
    return int(SequenceMatcher(None, token_sort_text(a), token_sort_text(b)).ratio() * 100)


def stable_hash_id(text: str, prefix: str) -> str:
    digest = hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]
    return f"{prefix}_{digest}"


def coalesce_columns(df: pd.DataFrame, columns: List[str]) -> pd.Series:
    values = []
    for col in columns:
        if col in df.columns:
            values.append(df[col])
    if not values:
        return pd.Series([pd.NA] * len(df), index=df.index)
    out = values[0].copy()
    for series in values[1:]:
        out = out.combine_first(series)
    return out


def ensure_columns(df: pd.DataFrame, required: List[str], table_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name}: missing required columns: {missing}")


def save_table(df: pd.DataFrame, name: str) -> None:
    parquet_path = OUTPUT_DIR / f"{name}.parquet"
    csv_path = OUTPUT_DIR / f"{name}.csv"
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)
    logger.info("Saved %s -> %s and %s", name, parquet_path.name, csv_path.name)


# FIXED: normalize nullable text fields to real Python None so Pydantic Optional[str]
# does not receive pd.NA / np.nan during row validation.
def to_python_none(value: Any) -> Any:
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip()
        return text if text else None
    return value


def normalize_optional_text_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    if column in df.columns:
        df[column] = df[column].map(to_python_none).astype("object")
    return df


def rename_columns_by_normalized_name(df: pd.DataFrame, normalized_map: Dict[str, str]) -> pd.DataFrame:
    rename_map: Dict[str, str] = {}
    for col in df.columns:
        key = normalize_column_name(col)
        if key in normalized_map:
            rename_map[col] = normalized_map[key]
    return df.rename(columns=rename_map)


def ascii_safe_column_name(name: Any) -> str:
    text = unicodedata.normalize("NFKD", str(name))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.replace("\xa0", " ").replace("\t", " ")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text or "col"


def strip_na_prefix(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = re.sub(r"^\s*NA\s*-\s*", "", str(text), flags=re.IGNORECASE).strip()
    return text if text else pd.NA


def build_synthetic_work_key(row: pd.Series) -> str:
    parts = [
        row.get("house"),
        row.get("state_key"),
        row.get("mp_key"),
        row.get("ida_key"),
        row.get("constituency"),
        row.get("work_category_raw"),
        row.get("work_description_clean"),
        row.get("work_raw"),
    ]
    seed = "|".join(
        str(part).strip().upper()
        for part in parts
        if pd.notna(part) and str(part).strip()
    )
    return stable_hash_id(seed, "wk")


def normalize_mp_name(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).upper()
    text = text.replace("\t", " ")
    text = HONORIFIC_RE.sub("", text)
    text = TRAILING_PARENS_RE.sub("", text)
    text = text.replace("&", " AND ")
    text = NON_ALNUM_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text if text else pd.NA


def extract_work_id_from_value(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).replace("\t", " ")
    match = WORK_ID_RE.search(text)
    if not match:
        return pd.NA
    return re.sub(r"\s+", "", match.group(0).upper())


def split_ida_value(value: Any) -> Tuple[Any, Any]:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA, pd.NA
    raw = str(text)
    district = raw.split("(", 1)[0].strip()
    district = WS_RE.sub(" ", district).strip()
    if not district:
        return pd.NA, pd.NA
    return district.title(), district.upper()


def normalize_vendor_name(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).upper()
    text = text.replace("&", " AND ")
    text = re.sub(r"\b(PVT|PRIVATE|LTD|LIMITED)\b", " ", text)
    text = re.sub(r"\bAND\b", " ", text)
    text = NON_ALNUM_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text if text else pd.NA

In [30]:
RAW_FILES = {
    "allocated_ls": DATA_DIR / "Allocated Limit for Honble MPs Lok Sabha.csv",
    "allocated_rs": DATA_DIR / "Allocated Limit for Honble MPs RajyaSabha.csv",
    "calamity_ls": DATA_DIR / "Amount consented for Calamity Lok Sabha .csv",
    "calamity_rs": DATA_DIR / "Amount consented for CalamityRajyaSabha.csv",
    "expenditure_ls": DATA_DIR / "Expenditure on Completed and On-going Works as on Date lok sabha.csv",
    "expenditure_rs": DATA_DIR / "Expenditure on Completed and On-going Works as on Date Rajya Sabha.csv",
    "state_wise": DATA_DIR / "MPLADS_State_Wise_Lok_Sabha_Rajya_Sabha.csv",
    "completed_ls": DATA_DIR / "Works Completed lok sabha.csv",
    "completed_rs": DATA_DIR / "Works Completed RAjyaSabha.csv",
    "recommended_ls": DATA_DIR / "Works Recommended lok sabha.csv",
    "recommended_rs": DATA_DIR / "Works Recommended RAjyaSabha.csv",
    "sanctioned_ls": DATA_DIR / "Works Sanctioned lok sabha.csv",
    "sanctioned_rs": DATA_DIR / "Works Sanctioned RajyaSabha.csv",
}


def read_csv_resilient(path: Path) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"]
    last_error: Optional[Exception] = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Failed to read {path}") from last_error


def load_all_raw_tables() -> Dict[str, pd.DataFrame]:
    tables: Dict[str, pd.DataFrame] = {}
    for key, path in RAW_FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing file: {path}")
        tables[key] = read_csv_resilient(path)
        logger.info("Loaded %s with shape %s", key, tables[key].shape)
    return tables


raw_tables = load_all_raw_tables()

In [31]:
ALLOCATED_ALIAS = {
    "srno": "sr_no",
    "state": "state",
    "honblemembersofparliaments": "mp_name_raw",
    "honblemembersofparliament": "mp_name_raw",
    "constituency": "constituency",
    "allocatedamount": "allocated_amount",
    "allocatedamountrs": "allocated_amount",
}

CALAMITY_ALIAS = {
    "srno": "sr_no",
    "calamitytype": "calamity_type",
    "calamityname": "calamity_name",
    "honblemembersofparliament": "mp_name_raw",
    "dateofconsent": "consent_date_raw",
    "consentamount": "consent_amount",
}

EXPENDITURE_ALIAS = {
    "srno": "sr_no",
    "state": "state",
    "work": "work_raw",
    "workid": "work_id_raw",
    "ida": "ida_raw",
    "honblemembersofparliament": "mp_name_raw",
    "constituency": "constituency",
    "electednominated": "elected_nominated",
    "expendituredate": "expenditure_date_raw",
    "vendorname": "vendor_raw",
    "paymentstatus": "payment_status",
    "funddisbursedamount": "fund_disbursed_amount",
}

STATEWISE_ALIAS = {
    "rank": "rank",
    "state": "state",
    "loksabhamps": "lok_sabha_mps",
    "loksabhaallocation": "lok_sabha_allocation",
    "loksabhavalidallocationrecords": "lok_sabha_valid_allocation_records",
    "loksabhaallocationcr": "lok_sabha_allocation_cr",
    "loksabhaaveragepervalidrecordcr": "lok_sabha_average_per_valid_record_cr",
    "rajasabhamps": "rajya_sabha_mps",
    "rajasabhaallocation": "rajya_sabha_allocation",
    "rajasabhavalidallocationrecords": "rajya_sabha_valid_allocation_records",
    "rajasabhaallocationcr": "rajya_sabha_allocation_cr",
    "rajasabhaaveragepervalidrecordcr": "rajya_sabha_average_per_valid_record_cr",
    "combinedmps": "combined_mps",
    "combinedallocation": "combined_allocation",
    "combinedallocationcr": "combined_allocation_cr",
}

WORK_ALIAS = {
    "srno": "sr_no",
    "workcategory": "work_category_raw",
    "workcategorys": "work_category_raw",
    "workcategoryo": "work_category_raw",
    "work": "work_raw",
    "state": "state",
    "ida": "ida_raw",
    "workdescription": "work_description_raw",
    "honblemembersofparliament": "mp_name_raw",
    "constituency": "constituency",
    "electednominated": "elected_nominated",
    "image": "image_raw",
    "completiondate": "completion_date_raw",
    "recommendeddate": "recommended_date_raw",
    "sanctiondate": "sanction_date_raw",
    "sanctionamount": "sanction_amount",
    "recommendedamount": "recommended_amount",
    "amountdisbursed": "amount_disbursed",
    "workstatus": "work_status",
}


def prep_allocated_table(df: pd.DataFrame, house: str) -> pd.DataFrame:
    df = standardize_columns(df, ALLOCATED_ALIAS)
    ensure_columns(df, ["sr_no", "state", "mp_name_raw", "allocated_amount"], f"allocated_{house}")
    df = drop_footer_rows(df, "sr_no")
    df = df.copy()
    df["house"] = house
    df["state"] = df["state"].map(normalize_state_name)
    df["state_key"] = df["state"]
    df["mp_name_raw"] = df["mp_name_raw"].map(clean_text)
    if "constituency" in df.columns:
        df["constituency"] = df["constituency"].map(clean_text)
    else:
        df["constituency"] = pd.NA
    # FIXED: coalesce the RS field from either raw header spelling, then drop the raw leak.
    if "Elected/Nominated" in df.columns:
        df["elected_nominated"] = coalesce_columns(df, ["elected_nominated", "Elected/Nominated"])
        df = df.drop(columns=["Elected/Nominated"], errors="ignore")
    elif "elected_nominated" in df.columns:
        df["elected_nominated"] = df["elected_nominated"].map(clean_text)
    else:
        df["elected_nominated"] = None
    # FIXED: remove any raw header variant after coalescing the clean field.
    raw_elected_cols = [c for c in df.columns if normalize_column_name(c) == "electednominated" and c != "elected_nominated"]
    df = df.drop(columns=raw_elected_cols, errors="ignore")
    df = normalize_optional_text_column(df, "constituency")
    df = normalize_optional_text_column(df, "elected_nominated")
    df["allocated_amount"] = df["allocated_amount"].map(parse_indian_amount)
    df["mp_name_clean"] = df["mp_name_raw"].map(normalize_mp_name)
    df["mp_key"] = df["house"] + "_" + df["mp_name_clean"].fillna("UNKNOWN")
    return df


def prep_calamity_table(df: pd.DataFrame, source_tag: str) -> pd.DataFrame:
    df = standardize_columns(df, CALAMITY_ALIAS)
    ensure_columns(df, ["sr_no", "calamity_type", "calamity_name", "mp_name_raw", "consent_date_raw", "consent_amount"], f"calamity_{source_tag}")
    df = drop_footer_rows(df, "sr_no")
    df = df.copy()
    # FIXED: the two calamity source files are identical, so the canonical deduped table
    # must not preserve conflicting LS/RS provenance labels.
    df["source_tag"] = "DEDUPED"
    df["calamity_type"] = df["calamity_type"].map(clean_text)
    df["calamity_name"] = df["calamity_name"].map(clean_text)
    df["mp_name_raw"] = df["mp_name_raw"].map(clean_text)
    df["mp_name_clean"] = df["mp_name_raw"].map(normalize_mp_name)
    df["consent_date"] = df["consent_date_raw"].map(parse_date)
    df["consent_amount"] = df["consent_amount"].map(parse_indian_amount)
    return df[[
        "source_tag",
        "calamity_type",
        "calamity_name",
        "mp_name_raw",
        "mp_name_clean",
        "consent_date",
        "consent_amount",
    ]].copy()


def prep_expenditure_table(df: pd.DataFrame, house: str) -> pd.DataFrame:
    df = standardize_columns(df, EXPENDITURE_ALIAS)
    ensure_columns(
        df,
        ["sr_no", "state", "work_raw", "work_id_raw", "ida_raw", "mp_name_raw", "expenditure_date_raw", "vendor_raw", "payment_status", "fund_disbursed_amount"],
        f"expenditure_{house}",
    )
    df = drop_footer_rows(df, "sr_no")
    df = df.copy()
    df["house"] = house
    df["state"] = df["state"].map(normalize_state_name)
    df["state_key"] = df["state"]
    df["work_raw"] = df["work_raw"].map(clean_text)
    df["work_id_raw"] = df["work_id_raw"].map(clean_text)
    df["work_id"] = df["work_id_raw"].map(extract_work_id_from_value)
    df["work_key"] = df["house"] + "|" + df["work_id"].fillna(df["work_id_raw"].fillna("UNKNOWN"))
    df["ida_raw"] = df["ida_raw"].map(clean_text)
    df[["district_name", "ida_key"]] = df["ida_raw"].apply(lambda x: pd.Series(split_ida_value(x)))
    df["mp_name_raw"] = df["mp_name_raw"].map(clean_text)
    df["mp_name_clean"] = df["mp_name_raw"].map(normalize_mp_name)
    df["mp_key"] = df["house"] + "_" + df["mp_name_clean"].fillna("UNKNOWN")
    if "constituency" in df.columns:
        df["constituency"] = df["constituency"].map(clean_text)
    else:
        df["constituency"] = pd.NA
    if "elected_nominated" in df.columns:
        df["elected_nominated"] = df["elected_nominated"].map(clean_text)
    else:
        df["elected_nominated"] = None
    # FIXED: normalize nullable text columns before validation / serialization.
    df = normalize_optional_text_column(df, "constituency")
    df = normalize_optional_text_column(df, "elected_nominated")
    df["expenditure_date"] = df["expenditure_date_raw"].map(parse_date)
    df["vendor_raw"] = df["vendor_raw"].map(clean_text)
    df["vendor_clean"] = df["vendor_raw"].map(normalize_vendor_name)
    df["payment_status"] = df["payment_status"].map(clean_text)
    df["fund_disbursed_amount"] = df["fund_disbursed_amount"].map(parse_indian_amount)
    return df


def prep_work_table(df: pd.DataFrame, house: str, stage: str) -> pd.DataFrame:
    df = standardize_columns(df, WORK_ALIAS)
    ensure_columns(df, ["sr_no", "work_raw", "state", "ida_raw", "mp_name_raw"], f"work_{stage}_{house}")
    df = drop_footer_rows(df, "sr_no")
    df = df.copy()
    df["source_stage"] = stage
    df["house"] = house
    df["state"] = df["state"].map(normalize_state_name)
    df["state_key"] = df["state"]
    # FIXED: strip the source-system "NA-" placeholder prefix from work titles.
    df["work_raw"] = df["work_raw"].map(strip_na_prefix)
    df["work_id"] = df["work_raw"].map(extract_work_id_from_value)
    df["ida_raw"] = df["ida_raw"].map(clean_text)
    df[["district_name", "ida_key"]] = df["ida_raw"].apply(lambda x: pd.Series(split_ida_value(x)))
    df["mp_name_raw"] = df["mp_name_raw"].map(clean_text)
    df["mp_name_clean"] = df["mp_name_raw"].map(normalize_mp_name)
    df["mp_key"] = df["house"] + "_" + df["mp_name_clean"].fillna("UNKNOWN")
    if "constituency" in df.columns:
        df["constituency"] = df["constituency"].map(clean_text)
    else:
        df["constituency"] = pd.NA
    if "elected_nominated" in df.columns:
        df["elected_nominated"] = df["elected_nominated"].map(clean_text)
    else:
        df["elected_nominated"] = None
    # FIXED: normalize optional text columns to Python None for downstream Pydantic.
    df = normalize_optional_text_column(df, "constituency")
    df = normalize_optional_text_column(df, "elected_nominated")
    if "work_category_raw" in df.columns:
        df["work_category_raw"] = df["work_category_raw"].map(clean_text)
    else:
        df["work_category_raw"] = pd.NA
    if "work_description_raw" in df.columns:
        df["work_description_raw"] = df["work_description_raw"].map(clean_text)
    else:
        df["work_description_raw"] = pd.NA
    if "image_raw" in df.columns:
        df["image_raw"] = df["image_raw"].map(clean_text)
    else:
        df["image_raw"] = pd.NA
    if "recommended_date_raw" in df.columns:
        df["recommended_date"] = df["recommended_date_raw"].map(parse_date)
    else:
        df["recommended_date"] = pd.NaT
    if "sanction_date_raw" in df.columns:
        df["sanction_date"] = df["sanction_date_raw"].map(parse_date)
    else:
        df["sanction_date"] = pd.NaT
    if "completion_date_raw" in df.columns:
        df["completion_date"] = df["completion_date_raw"].map(parse_date)
    else:
        df["completion_date"] = pd.NaT
    if "recommended_amount" in df.columns:
        df["recommended_amount"] = df["recommended_amount"].map(parse_indian_amount)
    else:
        df["recommended_amount"] = np.nan
    if "sanction_amount" in df.columns:
        df["sanction_amount"] = df["sanction_amount"].map(parse_indian_amount)
    else:
        df["sanction_amount"] = np.nan
    if "amount_disbursed" in df.columns:
        df["amount_disbursed"] = df["amount_disbursed"].map(parse_indian_amount)
    else:
        df["amount_disbursed"] = np.nan
    if "work_status" in df.columns:
        df["work_status"] = df["work_status"].map(clean_text)
    else:
        df["work_status"] = pd.NA
    df["category_raw"] = df["work_category_raw"]
    df["category_nlp"] = pd.NA
    df["work_description_clean"] = df["work_description_raw"].map(clean_text)
    # FIXED: do not leak "NA-..." into the canonical key; use a stable synthetic fallback
    # when the raw file does not provide a parseable Work ID.
    parsed_mask = df["work_id"].notna()
    df["work_key"] = pd.NA
    df.loc[parsed_mask, "work_key"] = df.loc[parsed_mask, "house"] + "|" + df.loc[parsed_mask, "work_id"].astype(str)
    df.loc[~parsed_mask, "work_key"] = df.loc[~parsed_mask].apply(build_synthetic_work_key, axis=1)
    return df

In [32]:
def normalize_mp_name(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).upper()
    text = text.replace("\t", " ")
    text = HONORIFIC_RE.sub("", text)
    text = TRAILING_PARENS_RE.sub("", text)
    text = text.replace("&", " AND ")
    text = NON_ALNUM_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text if text else pd.NA


def extract_work_id_from_value(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).replace("\t", " ")
    match = WORK_ID_RE.search(text)
    if not match:
        return pd.NA
    return re.sub(r"\s+", "", match.group(0).upper())


def split_ida_value(value: Any) -> Tuple[Any, Any]:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA, pd.NA
    raw = str(text)
    district = raw.split("(", 1)[0].strip()
    district = WS_RE.sub(" ", district).strip()
    if not district:
        return pd.NA, pd.NA
    return district.title(), district.upper()


def normalize_vendor_name(value: Any) -> Any:
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = str(text).upper()
    text = text.replace("&", " AND ")
    text = re.sub(r"\b(PVT|PRIVATE|LTD|LIMITED)\b", " ", text)
    text = re.sub(r"\bAND\b", " ", text)
    text = NON_ALNUM_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text if text else pd.NA


def build_state_lookup(all_states: Iterable[Any]) -> Dict[str, str]:
    cleaned = []
    for state in all_states:
        if pd.notna(state):
            cleaned_state = normalize_state_name(state)
            if pd.notna(cleaned_state):
                cleaned.append(cleaned_state)
    unique_states = sorted(set(cleaned))
    return {state: state for state in unique_states}


def resolve_state_key(value: Any, state_lookup: Dict[str, str]) -> Any:
    normalized = normalize_state_name(value)
    if pd.isna(normalized):
        return pd.NA
    return state_lookup.get(normalized, normalized)


def resolve_vendor_ids(df: pd.DataFrame, state_col: str = "state_key", vendor_col: str = "vendor_clean", threshold: int = 90) -> Tuple[pd.DataFrame, pd.DataFrame]:
    work = df.copy()

    resolution_rows = []
    mapping: Dict[Tuple[str, str], str] = {}

    grouped = work.dropna(subset=[vendor_col]).groupby(state_col, dropna=False, sort=True)
    for state_value, group in grouped:
        state_label = "UNKNOWN_STATE" if pd.isna(state_value) else str(state_value)
        unique_vendors = sorted(set(group[vendor_col].dropna().astype(str)))

        representatives: List[str] = []
        cluster_members: List[List[str]] = []

        for vendor in unique_vendors:
            assigned_cluster_idx = None
            for idx, rep in enumerate(representatives):
                score = token_sort_ratio(vendor, rep)
                if score >= threshold:
                    assigned_cluster_idx = idx
                    break

            if assigned_cluster_idx is None:
                representatives.append(vendor)
                cluster_members.append([vendor])
                assigned_cluster_idx = len(representatives) - 1
            else:
                cluster_members[assigned_cluster_idx].append(vendor)

        for idx, rep in enumerate(representatives):
            vendor_id = stable_hash_id(f"{state_label}|{rep}", "v")
            members = sorted(set(cluster_members[idx]))
            resolution_rows.append(
                {
                    "state_key": state_label,
                    "vendor_id": vendor_id,
                    "vendor_representative": rep,
                    "vendor_member_count": len(members),
                    "vendor_members": " || ".join(members),
                    "match_threshold": threshold,
                }
            )
            for member in members:
                mapping[(state_label, member)] = vendor_id

    work["vendor_id"] = [
        mapping.get((str(state_value) if pd.notna(state_value) else "UNKNOWN_STATE", str(vendor_value)))
        if pd.notna(vendor_value)
        else pd.NA
        for state_value, vendor_value in zip(work[state_col], work[vendor_col])
    ]

    vendor_resolution_table = pd.DataFrame(resolution_rows).sort_values(["state_key", "vendor_representative"]).reset_index(drop=True)
    return work, vendor_resolution_table

In [40]:
allocated_ls = prep_allocated_table(raw_tables["allocated_ls"], house="LS")
allocated_rs = prep_allocated_table(raw_tables["allocated_rs"], house="RS")
dim_mp = pd.concat([allocated_ls, allocated_rs], ignore_index=True, sort=False)
dim_mp = dim_mp.drop_duplicates(subset=["mp_key"]).reset_index(drop=True)
# FIXED: keep only the normalized elected_nominated column; drop any raw source leak.
if "Elected/Nominated" in dim_mp.columns:
    dim_mp["elected_nominated"] = coalesce_columns(dim_mp, ["elected_nominated", "Elected/Nominated"])
    dim_mp = dim_mp.drop(columns=["Elected/Nominated"], errors="ignore")


# 7.2 State-wise summary table
state_wise = standardize_columns(raw_tables["state_wise"], STATEWISE_ALIAS)
# FIXED: normalize every header to an ASCII-safe snake_case name before validation.
state_wise.columns = [ascii_safe_column_name(c) for c in state_wise.columns]
ensure_columns(state_wise, ["rank", "state"], "state_wise")
state_wise = drop_footer_rows(state_wise, "rank")
state_wise = state_wise.copy()
state_wise["state"] = state_wise["state"].map(normalize_state_name)
state_wise["state_key"] = state_wise["state"]
for col in state_wise.columns:
    if col not in {"rank", "state", "state_key"}:
        if "allocation" in col or "average" in col:
            state_wise[col] = state_wise[col].map(parse_indian_amount)
        else:
            state_wise[col] = pd.to_numeric(state_wise[col], errors="coerce")


# 7.3 Calamity tables
calamity_ls = prep_calamity_table(raw_tables["calamity_ls"], source_tag="DEDUPED")
calamity_rs = prep_calamity_table(raw_tables["calamity_rs"], source_tag="DEDUPED")

calamity_equal = raw_tables["calamity_ls"].equals(raw_tables["calamity_rs"])
logger.info("Calamity source files byte-for-byte equal: %s", calamity_equal)

fact_calamity = pd.concat([calamity_ls, calamity_rs], ignore_index=True, sort=False)
fact_calamity = fact_calamity.drop_duplicates(subset=["calamity_type", "calamity_name", "mp_name_raw", "consent_date", "consent_amount"]).reset_index(drop=True)
fact_calamity["source_tag"] = "DEDUPED"


# 7.4 Work-stage tables
recommended_ls = prep_work_table(raw_tables["recommended_ls"], house="LS", stage="recommended")
recommended_rs = prep_work_table(raw_tables["recommended_rs"], house="RS", stage="recommended")
sanctioned_ls = prep_work_table(raw_tables["sanctioned_ls"], house="LS", stage="sanctioned")
sanctioned_rs = prep_work_table(raw_tables["sanctioned_rs"], house="RS", stage="sanctioned")
completed_ls = prep_work_table(raw_tables["completed_ls"], house="LS", stage="completed")
completed_rs = prep_work_table(raw_tables["completed_rs"], house="RS", stage="completed")
expenditure_ls = prep_expenditure_table(raw_tables["expenditure_ls"], house="LS")
expenditure_rs = prep_expenditure_table(raw_tables["expenditure_rs"], house="RS")

recommended_all = pd.concat([recommended_ls, recommended_rs], ignore_index=True, sort=False)
sanctioned_all = pd.concat([sanctioned_ls, sanctioned_rs], ignore_index=True, sort=False)
completed_all = pd.concat([completed_ls, completed_rs], ignore_index=True, sort=False)
fact_expenditure = pd.concat([expenditure_ls, expenditure_rs], ignore_index=True, sort=False)

fact_expenditure = fact_expenditure.drop_duplicates(
    subset=["work_key", "expenditure_date", "vendor_raw", "fund_disbursed_amount", "payment_status"]
).reset_index(drop=True)


# 7.5 Harmonize work-stage frames into one long table and then pivot to the shared fact_work structure
work_stage_columns = [
    "work_key",
    "work_id",
    "house",
    "state",
    "state_key",
    "district_name",
    "ida_raw",
    "ida_key",
    "mp_name_raw",
    "mp_name_clean",
    "mp_key",
    "constituency",
    "elected_nominated",
    "source_stage",
    "work_raw",
    "work_category_raw",
    "category_raw",
    "category_nlp",
    "work_description_raw",
    "work_description_clean",
    "recommended_date",
    "sanction_date",
    "completion_date",
    "recommended_amount",
    "sanction_amount",
    "amount_disbursed",
    "work_status",
    "image_raw",
]


def align_stage_frame(df: pd.DataFrame, stage_name: str) -> pd.DataFrame:
    out = df.copy()
    out["source_stage"] = stage_name
    if "work_category_raw" not in out.columns:
        out["work_category_raw"] = pd.NA
    if "category_raw" not in out.columns:
        out["category_raw"] = out["work_category_raw"]
    if "image_raw" not in out.columns:
        out["image_raw"] = pd.NA
    if "work_description_clean" not in out.columns:
        out["work_description_clean"] = out.get("work_description_raw", pd.Series([pd.NA] * len(out)))
    for col in work_stage_columns:
        if col not in out.columns:
            out[col] = pd.NA
    return out[work_stage_columns].copy()


recommended_stage = align_stage_frame(recommended_all, "recommended")
sanctioned_stage = align_stage_frame(sanctioned_all, "sanctioned")
completed_stage = align_stage_frame(completed_all, "completed")

combined_work_stage = pd.concat([recommended_stage, sanctioned_stage, completed_stage], ignore_index=True, sort=False)

agg_map = {col: first_non_null for col in combined_work_stage.columns if col not in {"source_stage", "work_key"}}
agg_map["source_stage"] = unique_join

fact_work = (
    combined_work_stage.groupby("work_key", dropna=False, as_index=False)
    .agg(agg_map)
    .reset_index(drop=True)
)

# Prefer the most informative non-null stage values.
fact_work["work_raw"] = coalesce_columns(fact_work, ["work_raw"])
fact_work["category_raw"] = coalesce_columns(fact_work, ["category_raw", "work_category_raw"])
fact_work["work_description_raw"] = coalesce_columns(fact_work, ["work_description_raw"])
fact_work["work_description_clean"] = fact_work["work_description_raw"].map(clean_text)
fact_work["category_nlp"] = pd.NA  # populated later by Feature 3

# Clean up obvious placeholder columns.
for col in ["work_category_raw", "image_raw"]:
    if col in fact_work.columns:
        fact_work[col] = fact_work[col].map(clean_text)

# FIXED: keep canonical fact_work strictly at work_id grain.
# Rows without a parseable work_id are moved to a quarantine table so the main
# fact table matches the architecture and downstream joins stay deterministic.
fact_work_quarantine = fact_work[fact_work["work_id"].isna()].copy().reset_index(drop=True)
fact_work = fact_work[fact_work["work_id"].notna()].copy().reset_index(drop=True)
if len(fact_work_quarantine) > 0:
    fact_work_quarantine["quarantine_reason"] = "missing_parseable_work_id"
    fact_work_quarantine["work_key_mode"] = "synthetic"
    fact_work_quarantine["work_key_source"] = fact_work_quarantine["work_key"]
    logger.warning("Quarantined %s fact_work rows without parseable work_id", len(fact_work_quarantine))


# 7.6 Build a stable MP dimension from the cleaned allocation table and attach state lookup
state_lookup = build_state_lookup(
    pd.concat(
        [
            dim_mp.get("state", pd.Series(dtype="object")),
            state_wise.get("state", pd.Series(dtype="object")),
            fact_work.get("state", pd.Series(dtype="object")),
            fact_expenditure.get("state", pd.Series(dtype="object")),
        ],
        ignore_index=True,
    )
)

for table in [dim_mp, state_wise, fact_work, fact_expenditure]:
    if "state" in table.columns:
        table["state"] = table["state"].map(lambda x: resolve_state_key(x, state_lookup))
    if "state_key" in table.columns:
        table["state_key"] = table["state_key"].map(lambda x: resolve_state_key(x, state_lookup))


# 7.7 Re-resolve vendor IDs on the expenditure fact table
fact_expenditure, vendor_resolution_table = resolve_vendor_ids(fact_expenditure, state_col="state_key", vendor_col="vendor_clean", threshold=90)
fact_expenditure = fact_expenditure.drop(columns=["_vendor_cluster_seed", "_vendor_cluster_id"], errors="ignore")


# 7.8 Infer calamity mp house from the cleaned MP dimension when possible
mp_house_lookup = (
    dim_mp.dropna(subset=["mp_name_clean"])
    .groupby("mp_name_clean")["house"]
    .agg(lambda s: sorted(set(str(x) for x in s.dropna())))
    .to_dict()
)


def infer_calamity_house(mp_name_clean: Any) -> Any:
    if pd.isna(mp_name_clean):
        return pd.NA
    houses = mp_house_lookup.get(str(mp_name_clean), [])
    if len(houses) == 1:
        return houses[0]
    return "UNKNOWN"


fact_calamity["house"] = fact_calamity["mp_name_clean"].map(infer_calamity_house)
fact_calamity["mp_key"] = fact_calamity["house"].fillna("UNKNOWN") + "_" + fact_calamity["mp_name_clean"].fillna("UNKNOWN")


# 7.9 Create helper rollups that downstream features will reuse repeatedly
fact_expenditure_rollup = (
    fact_expenditure.groupby("work_key", as_index=False)
    .agg(
        work_id=("work_id", first_non_null),
        house=("house", first_non_null),
        state=("state", first_non_null),
        state_key=("state_key", first_non_null),
        mp_key=("mp_key", first_non_null),
        ida_key=("ida_key", first_non_null),
        district_name=("district_name", first_non_null),
        vendor_tranche_count=("vendor_id", "count"),
        n_vendors=("vendor_id", pd.Series.nunique),
        total_fund_disbursed=("fund_disbursed_amount", "sum"),
        first_expenditure_date=("expenditure_date", "min"),
        last_expenditure_date=("expenditure_date", "max"),
        any_success=("payment_status", lambda s: bool((s == "Payment Success").any())),
        any_in_progress=("payment_status", lambda s: bool((s == "Payment In-Progress").any())),
    )
    .reset_index(drop=True)
)

fact_work = fact_work.merge(
    fact_expenditure_rollup[["work_key", "vendor_tranche_count", "n_vendors", "total_fund_disbursed", "first_expenditure_date", "last_expenditure_date", "any_success", "any_in_progress"]],
    on="work_key",
    how="left",
)

fact_work["category_nlp"] = fact_work["category_nlp"].astype("object")

In [41]:
class DimMPRow(BaseModel):
    model_config = ConfigDict(extra="ignore", arbitrary_types_allowed=True)

    mp_key: str
    house: str
    state: Optional[str] = None
    state_key: Optional[str] = None
    mp_name_raw: Optional[str] = None
    mp_name_clean: Optional[str] = None
    constituency: Optional[str] = None
    elected_nominated: Optional[str] = None
    allocated_amount: Optional[float] = None


class FactWorkRow(BaseModel):
    model_config = ConfigDict(extra="ignore", arbitrary_types_allowed=True)

    work_key: str
    work_id: Optional[str] = None
    house: Optional[str] = None
    state: Optional[str] = None
    state_key: Optional[str] = None
    district_name: Optional[str] = None
    ida_raw: Optional[str] = None
    ida_key: Optional[str] = None
    mp_key: Optional[str] = None
    mp_name_raw: Optional[str] = None
    mp_name_clean: Optional[str] = None
    category_raw: Optional[str] = None
    category_nlp: Optional[str] = None
    work_description_raw: Optional[str] = None
    work_description_clean: Optional[str] = None
    recommended_date: Optional[pd.Timestamp] = None
    sanction_date: Optional[pd.Timestamp] = None
    completion_date: Optional[pd.Timestamp] = None
    recommended_amount: Optional[float] = None
    sanction_amount: Optional[float] = None
    amount_disbursed: Optional[float] = None
    work_status: Optional[str] = None
    image_raw: Optional[str] = None
    vendor_tranche_count: Optional[float] = None
    n_vendors: Optional[float] = None
    total_fund_disbursed: Optional[float] = None
    first_expenditure_date: Optional[pd.Timestamp] = None
    last_expenditure_date: Optional[pd.Timestamp] = None
    any_success: Optional[bool] = None
    any_in_progress: Optional[bool] = None


class FactExpenditureRow(BaseModel):
    model_config = ConfigDict(extra="ignore", arbitrary_types_allowed=True)

    work_key: str
    work_id: Optional[str] = None
    house: Optional[str] = None
    state: Optional[str] = None
    state_key: Optional[str] = None
    district_name: Optional[str] = None
    ida_raw: Optional[str] = None
    ida_key: Optional[str] = None
    mp_key: Optional[str] = None
    mp_name_raw: Optional[str] = None
    mp_name_clean: Optional[str] = None
    constituency: Optional[str] = None
    elected_nominated: Optional[str] = None
    expenditure_date: Optional[pd.Timestamp] = None
    vendor_raw: Optional[str] = None
    vendor_clean: Optional[str] = None
    vendor_id: Optional[str] = None
    payment_status: Optional[str] = None
    fund_disbursed_amount: Optional[float] = None


class FactCalamityRow(BaseModel):
    model_config = ConfigDict(extra="ignore", arbitrary_types_allowed=True)

    source_tag: str
    calamity_type: Optional[str] = None
    calamity_name: Optional[str] = None
    mp_name_raw: Optional[str] = None
    mp_name_clean: Optional[str] = None
    house: Optional[str] = None
    mp_key: Optional[str] = None
    consent_date: Optional[pd.Timestamp] = None
    consent_amount: Optional[float] = None


def validate_dataframe_rows(df: pd.DataFrame, model: type[BaseModel], table_name: str, max_errors: int = 20) -> None:
    errors = []
    for idx, record in enumerate(df.to_dict(orient="records")):
        try:
            # FIXED: sanitize row payloads before Pydantic so any lingering pd.NA/np.nan
            # values in optional text fields become plain Python None.
            record = {key: to_python_none(value) for key, value in record.items()}
            model.model_validate(record)
        except ValidationError as exc:
            errors.append((idx, str(exc)))
            if len(errors) >= max_errors:
                break
    if errors:
        preview = "\n".join([f"row={row_idx}: {msg}" for row_idx, msg in errors[:5]])
        raise ValueError(f"Validation failed for {table_name}. First errors:\n{preview}")
    logger.info("Validated %s rows for %s", len(df), table_name)


validate_dataframe_rows(dim_mp, DimMPRow, "dim_mp")
validate_dataframe_rows(fact_work, FactWorkRow, "fact_work")
validate_dataframe_rows(fact_expenditure, FactExpenditureRow, "fact_expenditure")
validate_dataframe_rows(fact_calamity, FactCalamityRow, "fact_calamity")


# Basic integrity checks specific to the shared layer.
# FIXED: extra guardrail for the column that triggered the notebook error.
for _df in (dim_mp, fact_work, fact_expenditure, fact_calamity):
    for _col in [c for c in _df.columns if c.endswith("_nominated") or c in {"constituency", "state", "state_key", "house", "mp_name_raw", "mp_name_clean", "district_name", "ida_raw", "ida_key", "vendor_raw", "vendor_clean", "work_status", "category_raw", "category_nlp", "work_description_raw", "work_description_clean", "payment_status", "calamity_type", "calamity_name", "source_tag"}]:
        _df[_col] = _df[_col].map(to_python_none).astype("object")
assert "Elected/Nominated" not in dim_mp.columns, "dim_mp still contains the raw RS allocation column"
assert dim_mp.loc[dim_mp["house"] == "RS", "elected_nominated"].notna().any(), "dim_mp RS rows lost elected/nominated values"
assert not any(any(ord(ch) > 127 for ch in str(col)) for col in state_wise.columns), "state_wise_summary still has non-ASCII headers"
assert not fact_work["work_key"].astype(str).str.contains(r"\|NA-", na=False).any(), "fact_work still leaks NA- into work_key"
assert fact_work["work_id"].notna().all(), "canonical fact_work still contains missing work_id rows"
assert fact_calamity["source_tag"].nunique() == 1 and fact_calamity["source_tag"].iloc[0] == "DEDUPED", "fact_calamity provenance label is inconsistent"
assert "_vendor_cluster_seed" not in fact_expenditure.columns and "_vendor_cluster_id" not in fact_expenditure.columns, "vendor helper columns should not be exported"
assert not dim_mp["mp_key"].isna().any(), "dim_mp contains null mp_key"
assert not fact_work["work_key"].isna().any(), "fact_work contains null work_key"
assert fact_calamity["consent_amount"].notna().sum() > 0, "fact_calamity consent amounts were not parsed"

logger.info("Shared preprocessing validation completed successfully.")

In [42]:
save_table(dim_mp, "dim_mp")
save_table(state_wise, "state_wise_summary")
save_table(fact_work, "fact_work")
save_table(fact_work_quarantine, "fact_work_quarantine")
save_table(fact_expenditure, "fact_expenditure")
save_table(fact_expenditure_rollup, "fact_expenditure_rollup")
save_table(fact_calamity, "fact_calamity")
save_table(vendor_resolution_table, "vendor_resolution_table")

artifact_index = {
    "dim_mp": str(OUTPUT_DIR / "dim_mp.parquet"),
    "state_wise_summary": str(OUTPUT_DIR / "state_wise_summary.parquet"),
    "fact_work": str(OUTPUT_DIR / "fact_work.parquet"),
    "fact_work_quarantine": str(OUTPUT_DIR / "fact_work_quarantine.parquet"),
    "fact_expenditure": str(OUTPUT_DIR / "fact_expenditure.parquet"),
    "fact_expenditure_rollup": str(OUTPUT_DIR / "fact_expenditure_rollup.parquet"),
    "fact_calamity": str(OUTPUT_DIR / "fact_calamity.parquet"),
    "vendor_resolution_table": str(OUTPUT_DIR / "vendor_resolution_table.parquet"),
}

print("Saved artifacts:")
for k, v in artifact_index.items():
    print(f"- {k}: {v}")

Saved artifacts:
- dim_mp: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/dim_mp.parquet
- state_wise_summary: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/state_wise_summary.parquet
- fact_work: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/fact_work.parquet
- fact_work_quarantine: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/fact_work_quarantine.parquet
- fact_expenditure: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/fact_expenditure.parquet
- fact_expenditure_rollup: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/fact_expenditure_rollup.parquet
- fact_calamity: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/fact_calamity.parquet
- vendor_resolution_table: /content/drive/MyDrive/MPLADs/shared_preprocessing_artifacts/vendor_resolution_table.parquet
